# Data audit

Scan recordings, report sample rate, duration, channels and produce a small CSV summary.
Run this after you place files in data/raw/recordings or data/processed/wavs.


In [ ]:
from pathlib import Path

import soundfile as sf

EXTS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}
RAW_DIR = Path("data/raw/recordings")
PROC_DIR = Path("data/processed/wavs")


def scan_folder(folder: Path):
    rows = []
    if not folder.exists():
        return rows
    for file in sorted(folder.rglob("*")):
        if file.suffix.lower() not in EXTS:
            continue
        try:
            info = sf.info(str(file))
            sample_rate = int(info.sample_rate) if info.sample_rate else None
            frames = int(info.frames) if info.frames else None
            duration = (
                round(frames / sample_rate, 3) if frames and sample_rate else None
            )
            channels = int(info.channels) if info.channels else None
            rows.append(
                {
                    "path": str(file),
                    "sample_rate": sample_rate,
                    "duration_s": duration,
                    "channels": channels,
                }
            )
        except Exception as error:
            rows.append(
                {
                    "path": str(file),
                    "sample_rate": None,
                    "duration_s": None,
                    "channels": None,
                    "error": str(error),
                }
            )
    return rows

## Let's output to a CSV

What it does: runs scan_folder on RAW_DIR and PROC_DIR, computes count and duration stats, prints sample-rate and channel distributions, writes CSV summary to data/outputs/data_audit_summary.csv.

In [ ]:
import csv
import statistics
from pathlib import Path

out_dir = Path("data/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "data_audit_summary.csv"

raw_rows = scan_folder(RAW_DIR)
proc_rows = scan_folder(PROC_DIR)
all_rows = raw_rows + proc_rows

print(
    f"Found {len(raw_rows)} in {RAW_DIR}, {len(proc_rows)} in {PROC_DIR}, total {len(all_rows)}"
)

durations = [
    row["duration_s"]
    for row in all_rows
    if isinstance(row.get("duration_s"), (int, float))
]

if durations:
    print(
        "Duration (s) min/max/mean:",
        min(durations),
        max(durations),
        round(statistics.mean(durations), 3),
    )
else:
    print("No valid durations found.")

sample_rates = {}

for row in all_rows:
    sample_rate = row.get("sample_rate")
    sample_rates[sample_rate] = sample_rates.get(sample_rate, 0) + 1
print("Sample rates:", sample_rates)

channels = {}

for row in all_rows:
    channel = row.get("channels")
    channels[channel] = channels.get(channel, 0) + 1
print("Channels:", channels)

with open(csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["path", "sample_rate", "channels", "duration_s", "error"])
    for row in all_rows:
        writer.writerow(
            [
                row.get("path"),
                row.get("sample_rate"),
                row.get("channels"),
                row.get("duration_s"),
                row.get("error", ""),
            ]
        )
print("Wrote summary to", csv_path)

## Interpretation and next steps

- If the sample rates vary -> resample to target SR in preprocessing (e.g., 22050)
- If channels > 1 -> convert to mono
- Very short (<0.2s) or very long (>20s) files -> inspect/remove/split
- Fix errors reported (codec issues) by re-encoding or re-exporting